# A Lightweight eRAG-Driven Experience-Graph Agentic Framework
## Behavioural Drift and Decision Contagion Detection in Autonomous Financial Agents

This notebook implements the full 7-phase pipeline described in the paper spec:

1. **Phase 1 — FinState-Lite**: Data preprocessing & compact financial state construction
2. **Phase 2 — eRAG-ReflectX**: Regime-adaptive experience retrieval & memory contagion modelling
3. **Phase 3 — DA-Delib-Mix4**: Drift-adaptive heterogeneous multi-agent decision formation
4. **Phase 4 — TempRel-Lite**: Temporal behaviour interaction (graph) modelling
5. **Phase 5 — DriftCont-X**: Behavioural drift & decision contagion detection
6. **Phase 6 — ReflectAdapt-X**: Self-reflection, memory evolution & policy adaptation (closes the loop)
7. **Phase 7 — ConforRisk-X**: Uncertainty-aware risk orchestration → **AFBRI** (Autonomous Financial Behaviour Risk Index)

**Data source**: designed to plug into raw observations/actions/portfolio logs from
[`HKUDS/AI-Trader`](https://github.com/HKUDS/AI-Trader). Since this notebook runs offline,
Section 0 includes a **synthetic AI-Trader-style data generator** that reproduces the same
schema (market observations, agent actions, portfolio records, timestamps) so the full
pipeline can be demonstrated end-to-end. Swap `load_ai_trader_data()` for a real loader that
reads the cloned repo's logs/CSV/JSON exports and everything downstream is unchanged.

Every phase is implemented as an independent, testable class so it can be swapped, extended,
or replaced with a learned model later (e.g. the "Lightweight Vector Retrieval" module here
uses cosine similarity over hand-built features; you could later replace it with a learned
encoder without touching Phases 3-7).


## 0. Setup & Imports

In [ ]:

import json
import math
import random
import warnings
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional, Any
from collections import deque, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
RNG = np.random.default_rng(42)
random.seed(42)

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


## 0.1 Synthetic AI-Trader-style Data Generator

Reproduces the schema you'd get from `AI-Trader` logs: per-timestamp market observations,
per-agent actions, portfolio state, and outcomes. Four agent archetypes are simulated with
a deliberately injected **regime shift** and one agent (`Momentum-Chaser`) that undergoes a
**behavioural drift** halfway through (risk appetite creeps up) which should later propagate
(contagion) into a second agent that imitates it. This gives the drift/contagion detectors
in Phase 5 something real to find.

Replace this cell with a loader against your cloned `AI-Trader` repo output when available —
downstream phases only depend on the columns defined in `RAW_COLUMNS` below.


In [ ]:

RAW_COLUMNS = [
    "timestamp", "agent_id", "price", "volume", "volatility", "market_regime",
    "action_type", "position_size", "cash", "holdings_value", "portfolio_value",
    "pnl", "confidence_self_reported",
]

AGENTS = ["Market-Agent", "Risk-Agent", "Portfolio-Agent", "Challenge-Agent"]
ACTIONS = ["BUY", "SELL", "HOLD", "REDUCE", "HEDGE"]
REGIMES = ["low_vol_bull", "high_vol_bear", "sideways_chop", "trending_recovery"]

def load_ai_trader_data(n_steps: int = 240, drift_start_frac: float = 0.55,
                         contagion_agent: str = "Risk-Agent",
                         drifting_agent: str = "Portfolio-Agent") -> pd.DataFrame:
    """Synthetic stand-in for reading AI-Trader raw logs (see github.com/HKUDS/AI-Trader).
    Produces one row per (timestamp, agent) with the RAW_COLUMNS schema."""
    rows = []
    base_price = 100.0
    price = base_price
    regime_boundaries = sorted(RNG.choice(range(20, n_steps - 20), size=3, replace=False))
    drift_step = int(n_steps * drift_start_frac)

    portfolio_state = {a: {"cash": 100_000.0, "position_size": 0.0, "holdings_value": 0.0}
                        for a in AGENTS}

    for t in range(n_steps):
        regime_idx = sum(t > b for b in regime_boundaries) % len(REGIMES)
        regime = REGIMES[regime_idx]
        vol = {"low_vol_bull": 0.4, "high_vol_bear": 2.2,
               "sideways_chop": 0.9, "trending_recovery": 1.2}[regime]
        drift_component = {"low_vol_bull": 0.15, "high_vol_bear": -0.25,
                            "sideways_chop": 0.0, "trending_recovery": 0.2}[regime]
        price = max(1.0, price * (1 + RNG.normal(drift_component / 100, vol / 100)))
        volume = max(1000, RNG.normal(50_000, 15_000))
        ts = pd.Timestamp("2025-01-01") + pd.Timedelta(hours=t)

        for agent in AGENTS:
            drifted = (agent == drifting_agent and t >= drift_step)
            contagious = (agent == contagion_agent and t >= drift_step + 12)

            risk_appetite = 0.5
            if drifted:
                progress = min(1.0, (t - drift_step) / max(1, n_steps - drift_step))
                risk_appetite = 0.5 + 0.45 * progress
            if contagious:
                progress = min(1.0, (t - drift_step - 12) / max(1, n_steps - drift_step - 12))
                risk_appetite = 0.5 + 0.30 * progress

            action_logits = RNG.normal(0, 1, size=len(ACTIONS))
            if risk_appetite > 0.6:
                action_logits[ACTIONS.index("BUY")] += 2.0 * risk_appetite
                action_logits[ACTIONS.index("HEDGE")] -= 1.0
            action = ACTIONS[int(np.argmax(action_logits))]

            size_delta = {"BUY": 1, "SELL": -1, "REDUCE": -0.5, "HEDGE": -0.3, "HOLD": 0}[action]
            size_delta *= (500 + 400 * risk_appetite)
            ps = portfolio_state[agent]
            ps["position_size"] = max(0.0, ps["position_size"] + size_delta)
            ps["holdings_value"] = ps["position_size"] * price
            trade_cost = size_delta * price
            ps["cash"] -= trade_cost
            portfolio_value = ps["cash"] + ps["holdings_value"]
            pnl = portfolio_value - 100_000.0

            confidence = float(np.clip(RNG.normal(0.55 + 0.2 * risk_appetite, 0.1), 0.05, 0.99))

            rows.append(dict(
                timestamp=ts, agent_id=agent, price=round(price, 4), volume=round(volume, 1),
                volatility=round(vol + RNG.normal(0, 0.05), 4), market_regime=regime,
                action_type=action, position_size=round(ps["position_size"], 2),
                cash=round(ps["cash"], 2), holdings_value=round(ps["holdings_value"], 2),
                portfolio_value=round(portfolio_value, 2), pnl=round(pnl, 2),
                confidence_self_reported=round(confidence, 4),
            ))

    df = pd.DataFrame(rows, columns=RAW_COLUMNS)
    df.attrs["drift_step"] = drift_step
    df.attrs["drifting_agent"] = drifting_agent
    df.attrs["contagion_agent"] = contagion_agent
    return df

raw_df = load_ai_trader_data()
print(f"Loaded {len(raw_df)} rows | {raw_df['agent_id'].nunique()} agents | "
      f"{raw_df['timestamp'].nunique()} timestamps")
raw_df.head()


---
## Phase 1 — 3.3 Lightweight Data Preprocessing and Financial State Construction

**Input**: raw market observations, agent actions, portfolio records, timestamps.
**Steps**: (1) temporal alignment & cleaning, (2) normalization & categorical encoding,
(3) decision-episode construction (causal, no leakage), (4) FinState-Lite feature
engineering, (5) temporal-difference encoding, (6) adaptive feature gating.
**Output**: compact financial state `S_t` per (agent, timestamp).


In [ ]:

@dataclass
class FinancialState:
    agent_id: str
    timestamp: pd.Timestamp
    step_idx: int
    market_regime: str
    action_type: str
    features: np.ndarray          # gated FinState-Lite feature vector
    raw_features: np.ndarray      # pre-gating feature vector (for TD-encoding, retrieval)
    feature_names: List[str]
    confidence: float
    outcome: Optional[float] = None   # filled in once next-step pnl delta is known


class FinStateLitePreprocessor:
    """Phase 1: Steps 1-6. Leakage-safe: every feature at step t only uses data <= t."""

    CONTINUOUS_COLS = ["price", "volume", "volatility", "position_size", "cash",
                        "holdings_value", "portfolio_value", "pnl", "confidence_self_reported"]

    def __init__(self):
        self.scalers: Dict[str, StandardScaler] = {}
        self.action_vocab = {a: i for i, a in enumerate(ACTIONS)}
        self.regime_vocab = {r: i for i, r in enumerate(REGIMES)}
        self._fitted = False

    # Step 1 -----------------------------------------------------------
    def clean_and_align(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop_duplicates(subset=["agent_id", "timestamp"]).copy()
        df = df.sort_values(["agent_id", "timestamp"]).reset_index(drop=True)
        # leakage-safe interpolation: forward-fill only (never uses future rows)
        df[self.CONTINUOUS_COLS] = (
            df.groupby("agent_id")[self.CONTINUOUS_COLS]
              .apply(lambda g: g.ffill())   # leakage-safe: only ever fills forward from the past
              .reset_index(drop=True)
        )
        df = df.dropna(subset=self.CONTINUOUS_COLS)
        return df

    # Step 2 -----------------------------------------------------------
    def fit_normalizers(self, train_df: pd.DataFrame):
        """Fit Z-score stats on a designated TRAIN split only (no leakage into test)."""
        for agent, g in train_df.groupby("agent_id"):
            scaler = StandardScaler()
            scaler.fit(g[self.CONTINUOUS_COLS].values)
            self.scalers[agent] = scaler
        self._fitted = True

    def normalize_and_encode(self, df: pd.DataFrame) -> pd.DataFrame:
        assert self._fitted, "Call fit_normalizers() on a train split first."
        out = []
        for agent, g in df.groupby("agent_id"):
            scaler = self.scalers.get(agent)
            if scaler is None:
                scaler = StandardScaler().fit(g[self.CONTINUOUS_COLS].values)
            z = scaler.transform(g[self.CONTINUOUS_COLS].values)
            g = g.copy()
            for i, c in enumerate(self.CONTINUOUS_COLS):
                g[f"z_{c}"] = z[:, i]
            g["action_code"] = g["action_type"].map(self.action_vocab)
            g["regime_code"] = g["market_regime"].map(self.regime_vocab)
            out.append(g)
        return pd.concat(out).sort_values(["agent_id", "timestamp"]).reset_index(drop=True)

    # Step 3 -----------------------------------------------------------
    def build_decision_episodes(self, df: pd.DataFrame) -> pd.DataFrame:
        """Each row already IS a decision episode (one action per agent per timestamp);
        here we just assign a monotonically increasing per-agent step index used causally
        downstream (windowed features only look back)."""
        df = df.copy()
        df["step_idx"] = df.groupby("agent_id").cumcount()
        return df

    # Step 4 -----------------------------------------------------------
    def engineer_finstate_lite(self, df: pd.DataFrame, window: int = 5) -> pd.DataFrame:
        """Market, volatility/risk, portfolio, action-transition, temporal-change features."""
        df = df.copy()
        g = df.groupby("agent_id")

        # market features
        df["mkt_price_z"] = df["z_price"]
        df["mkt_vol_z"] = df["z_volume"]

        # volatility / risk features
        df["risk_volatility_z"] = df["z_volatility"]
        df["risk_rolling_vol"] = g["z_price"].transform(
            lambda s: s.rolling(window, min_periods=1).std().fillna(0))

        # portfolio features
        df["port_position_z"] = df["z_position_size"]
        df["port_value_z"] = df["z_portfolio_value"]
        df["port_pnl_z"] = df["z_pnl"]
        df["port_pnl_rolling_mean"] = g["z_pnl"].transform(
            lambda s: s.rolling(window, min_periods=1).mean())

        # action-transition features: has the action changed vs previous step?
        df["action_prev"] = g["action_code"].shift(1).fillna(df["action_code"])
        df["action_changed"] = (df["action_code"] != df["action_prev"]).astype(float)
        df["action_switch_rate"] = g["action_changed"].transform(
            lambda s: s.rolling(window, min_periods=1).mean())

        # confidence
        df["conf_z"] = df["z_confidence_self_reported"]

        self.finstate_cols = [
            "mkt_price_z", "mkt_vol_z", "risk_volatility_z", "risk_rolling_vol",
            "port_position_z", "port_value_z", "port_pnl_z", "port_pnl_rolling_mean",
            "action_changed", "action_switch_rate", "conf_z",
        ]
        return df

    # Step 5 -----------------------------------------------------------
    def temporal_difference_encode(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        g = df.groupby("agent_id")
        for c in self.finstate_cols:
            df[f"td_{c}"] = g[c].diff().fillna(0.0)
        self.td_cols = [f"td_{c}" for c in self.finstate_cols]
        return df

    # Step 6 -----------------------------------------------------------
    def adaptive_feature_gate(self, row: pd.Series) -> np.ndarray:
        """Regime-conditional soft gate: up-weights the feature groups that matter most
        in the current market regime (e.g. volatility features matter more in high-vol
        regimes; portfolio/pnl features matter more in trending/recovery regimes)."""
        base = np.array([row[c] for c in self.finstate_cols], dtype=float)
        gate = np.ones_like(base)
        regime = row["market_regime"]
        idx = {name: i for i, name in enumerate(self.finstate_cols)}
        if regime == "high_vol_bear":
            for k in ["risk_volatility_z", "risk_rolling_vol", "action_switch_rate"]:
                gate[idx[k]] *= 1.6
        elif regime == "low_vol_bull":
            for k in ["port_pnl_z", "port_pnl_rolling_mean", "port_value_z"]:
                gate[idx[k]] *= 1.5
        elif regime == "sideways_chop":
            for k in ["action_changed", "action_switch_rate"]:
                gate[idx[k]] *= 1.4
        elif regime == "trending_recovery":
            for k in ["port_pnl_rolling_mean", "mkt_price_z"]:
                gate[idx[k]] *= 1.3
        return base * gate

    # orchestration ------------------------------------------------------
    def run(self, raw_df: pd.DataFrame, train_frac: float = 0.5) -> List[FinancialState]:
        df = self.clean_and_align(raw_df)

        split_t = df["timestamp"].quantile(train_frac)
        train_df = df[df["timestamp"] <= split_t]
        self.fit_normalizers(train_df)

        df = self.normalize_and_encode(df)
        df = self.build_decision_episodes(df)
        df = self.engineer_finstate_lite(df)
        df = self.temporal_difference_encode(df)

        states = []
        feat_names = self.finstate_cols + self.td_cols
        for _, row in df.iterrows():
            raw_vec = np.array([row[c] for c in self.finstate_cols] +
                                [row[c] for c in self.td_cols], dtype=float)
            gated_base = self.adaptive_feature_gate(row)
            gated_vec = np.concatenate([gated_base, np.array([row[c] for c in self.td_cols])])
            states.append(FinancialState(
                agent_id=row["agent_id"], timestamp=row["timestamp"], step_idx=int(row["step_idx"]),
                market_regime=row["market_regime"], action_type=row["action_type"],
                features=gated_vec, raw_features=raw_vec, feature_names=feat_names,
                confidence=float(row["confidence_self_reported"]),
            ))
        # attach next-step pnl delta as the realised outcome (causal: known only after the fact)
        outcomes = df.groupby("agent_id")["pnl"].diff().shift(-1).fillna(0.0).values
        for s, o in zip(states, outcomes):
            s.outcome = float(o)
        return states


preprocessor = FinStateLitePreprocessor()
states = preprocessor.run(raw_df)
print(f"Built {len(states)} FinState-Lite states | feature dim = {states[0].features.shape[0]}")
pd.DataFrame([{"agent": s.agent_id, "t": s.step_idx, "regime": s.market_regime,
               "action": s.action_type, "outcome": round(s.outcome, 2)} for s in states[:6]])


---
## Phase 2 — 3.4 eRAG-ReflectX: Regime-Adaptive Experience Retrieval & Memory Contagion

Historical episodes `e_i = [S_i ‖ A_i ‖ O_i ‖ C_i ‖ R_i]` are stored in an episodic memory.
- **Episodic Memory Builder** — stores structured episodes.
- **Lightweight Vector Retrieval** — cosine similarity over `ψ(S)` (the gated feature vector).
- **Semantic Prototype Memory** — per-regime centroid prototypes for fast regime matching.
- **Relevance Gate** — combines state similarity, regime match, outcome relevance, reflection quality.

`MRD_t = 1 - (1/k) Σ sim(S_t, S_i)` over the retrieved top-k (**Memory Retrieval Drift**):
high MRD means the agent is operating somewhere its memory has little useful precedent for.


In [ ]:

@dataclass
class MemoryRecord:
    state: FinancialState
    action: str
    outcome: float
    confidence: float
    reflection: str = ""
    reflection_quality: float = 0.5    # updated by Phase 6


class EpisodicMemoryBuilder:
    def __init__(self):
        self.records: List[MemoryRecord] = []

    def add(self, rec: MemoryRecord):
        self.records.append(rec)

    def add_state(self, state: FinancialState):
        self.add(MemoryRecord(state=state, action=state.action_type,
                               outcome=state.outcome, confidence=state.confidence))


class SemanticPrototypeMemory:
    """Maintains a running centroid feature vector per market regime for fast regime-level
    similarity ('does this look like a high_vol_bear situation overall?')."""
    def __init__(self):
        self.prototypes: Dict[str, np.ndarray] = {}
        self.counts: Dict[str, int] = defaultdict(int)

    def update(self, regime: str, vec: np.ndarray):
        n = self.counts[regime]
        if regime not in self.prototypes:
            self.prototypes[regime] = vec.copy()
        else:
            self.prototypes[regime] = (self.prototypes[regime] * n + vec) / (n + 1)
        self.counts[regime] += 1

    def regime_similarity(self, regime: str, vec: np.ndarray) -> float:
        proto = self.prototypes.get(regime)
        if proto is None:
            return 0.0
        return float(cosine_similarity(vec.reshape(1, -1), proto.reshape(1, -1))[0, 0])


class eRAGReflectX:
    """Phase 2 orchestration: memory builder + retrieval + prototypes + relevance gate."""

    def __init__(self, top_k: int = 8,
                 w_state: float = 0.45, w_regime: float = 0.20,
                 w_outcome: float = 0.20, w_reflection: float = 0.15):
        self.memory = EpisodicMemoryBuilder()
        self.prototypes = SemanticPrototypeMemory()
        self.top_k = top_k
        self.weights = dict(state=w_state, regime=w_regime, outcome=w_outcome, reflection=w_reflection)

    def _vector_retrieval(self, query: np.ndarray, candidates: List[MemoryRecord]) -> np.ndarray:
        if not candidates:
            return np.array([])
        mat = np.stack([c.state.features for c in candidates])
        return cosine_similarity(query.reshape(1, -1), mat)[0]

    def _relevance_gate(self, state_sim: np.ndarray, candidates: List[MemoryRecord],
                         current_regime: str) -> np.ndarray:
        regime_sim = np.array([1.0 if c.state.market_regime == current_regime else 0.3
                                for c in candidates])
        z = np.clip(np.array([c.outcome for c in candidates]) / 50, -30, 30)
        outcome_rel = 1 / (1 + np.exp(-z))  # squashed pnl, overflow-safe
        reflection_q = np.array([c.reflection_quality for c in candidates])
        w = self.weights
        score = (w["state"] * state_sim + w["regime"] * regime_sim +
                 w["outcome"] * outcome_rel + w["reflection"] * reflection_q)
        return score

    def retrieve(self, query_state: FinancialState) -> Tuple[List[MemoryRecord], float]:
        """Returns (top-k experience-guided decision context, Memory Retrieval Drift)."""
        candidates = [r for r in self.memory.records if r.state.agent_id == query_state.agent_id]
        if not candidates:
            return [], 1.0   # no memory at all => max drift (nothing to retrieve from)

        state_sim = self._vector_retrieval(query_state.features, candidates)
        gated_score = self._relevance_gate(state_sim, candidates, query_state.market_regime)

        order = np.argsort(-gated_score)[: self.top_k]
        top_candidates = [candidates[i] for i in order]
        top_sims = state_sim[order]

        mrd = float(1 - np.mean(top_sims)) if len(top_sims) else 1.0
        return top_candidates, mrd

    def observe(self, state: FinancialState):
        """Call once a state (and its realised outcome) is known, to grow the memory."""
        self.memory.add_state(state)
        self.prototypes.update(state.market_regime, state.features)


erag = eRAGReflectX(top_k=8)
mrd_series = []
experience_contexts = {}   # step_idx -> {agent: [MemoryRecord,...]}

for s in states:
    ctx, mrd = erag.retrieve(s)
    experience_contexts[(s.agent_id, s.step_idx)] = ctx
    mrd_series.append(dict(agent=s.agent_id, step=s.step_idx, mrd=mrd, regime=s.market_regime))
    erag.observe(s)   # grow memory causally, one step at a time (no lookahead)

mrd_df = pd.DataFrame(mrd_series)
mrd_df.groupby("agent")["mrd"].mean().rename("avg_MRD").to_frame()


---
## Phase 3 — 3.5 DA-Delib-Mix4: Drift-Adaptive Heterogeneous Multi-Agent Decision Formation

Four lightweight agents (**Market**, **Risk**, **Portfolio**, **Challenge**) each independently
propose `(action, confidence, decision representation)` from the same state + experience
context. **Agreement/Disagreement Analysis** compares them; the **Confidence Mixer** blends
recommendations by confidence × historical reliability × contextual relevance;
**Dynamic Challenge Routing** escalates review when disagreement crosses a threshold.

Output: collective decision state `C_t = [A_t ‖ C_t^conf ‖ Agr_t ‖ Dis_t ‖ Inf_t]`.


In [ ]:

@dataclass
class AgentProposal:
    agent_role: str
    action: str
    confidence: float
    decision_vec: np.ndarray   # concise decision representation


@dataclass
class CollectiveDecisionState:
    step_idx: int
    subject_agent: str
    final_action: str
    collective_confidence: float
    agreement: float
    disagreement: float
    influence: Dict[str, float]         # per-role influence weight on final decision
    challenge_triggered: bool
    proposals: List[AgentProposal]


class HeterogeneousAgentDeliberation:
    """Each role has a lightweight rule-of-thumb policy over (state, experience context).
    In a full system these would be separate learned/LLM policies; here they are transparent
    heuristics so the pipeline is fully runnable offline, and each is easily swappable."""

    ROLE_BIAS = {
        # role -> (action it leans toward when risk_signal high, weight on risk feature)
        "Market-Agent": ("BUY", 0.6),
        "Risk-Agent": ("HEDGE", 1.4),
        "Portfolio-Agent": ("REDUCE", 1.0),
        "Challenge-Agent": ("SELL", 1.2),
    }

    def propose(self, role: str, state: FinancialState, ctx: List[MemoryRecord]) -> AgentProposal:
        idx = {n: i for i, n in enumerate(state.feature_names)}
        risk_signal = state.features[idx["risk_volatility_z"]] + state.features[idx["risk_rolling_vol"]]
        pnl_signal = state.features[idx["port_pnl_rolling_mean"]]
        switch_signal = state.features[idx["action_switch_rate"]]

        exp_action_bias = 0.0
        if ctx:
            # what did similar past experiences do, and how did it turn out?
            good = [c for c in ctx if c.outcome > 0]
            exp_action_bias = len(good) / len(ctx) - 0.5   # in [-0.5, 0.5]

        lean_action, w = self.ROLE_BIAS[role]
        score = w * risk_signal - 0.5 * pnl_signal + 0.3 * switch_signal + exp_action_bias
        score += RNG.normal(0, 0.15)   # small stochastic exploration

        if score > 0.8:
            action = lean_action
        elif score < -0.8:
            action = "BUY" if lean_action != "BUY" else "HOLD"
        else:
            action = "HOLD"

        confidence = float(np.clip(0.5 + 0.25 * abs(score) + 0.1 * (len(ctx) > 0), 0.05, 0.99))
        decision_vec = np.array([score, risk_signal, pnl_signal, switch_signal, exp_action_bias])
        return AgentProposal(agent_role=role, action=action, confidence=confidence, decision_vec=decision_vec)


class AgreementDisagreementAnalysis:
    def analyze(self, proposals: List[AgentProposal]) -> Tuple[float, float, np.ndarray]:
        actions = [p.action for p in proposals]
        most_common = max(set(actions), key=actions.count)
        agreement = actions.count(most_common) / len(actions)
        disagreement = 1 - agreement
        vecs = np.stack([p.decision_vec for p in proposals])
        sim_matrix = cosine_similarity(vecs)
        return agreement, disagreement, sim_matrix


class ConfidenceMixer:
    def __init__(self):
        self.reliability: Dict[str, float] = {r: 0.5 for r in AGENTS}   # updated by Phase 6

    def mix(self, proposals: List[AgentProposal], sim_matrix: np.ndarray) -> Tuple[str, float, Dict[str, float]]:
        contextual_relevance = sim_matrix.mean(axis=1)   # how central each proposal is
        weights = np.array([
            p.confidence * self.reliability[p.agent_role] * contextual_relevance[i]
            for i, p in enumerate(proposals)
        ])
        weights = weights / (weights.sum() + 1e-9)

        action_scores = defaultdict(float)
        for p, w in zip(proposals, weights):
            action_scores[p.action] += w
        final_action = max(action_scores, key=action_scores.get)
        collective_confidence = float(sum(w * p.confidence for w, p in zip(weights, proposals)))
        influence = {p.agent_role: float(w) for p, w in zip(proposals, weights)}
        return final_action, collective_confidence, influence


class DynamicChallengeRouting:
    def __init__(self, disagreement_threshold: float = 0.5):
        self.threshold = disagreement_threshold

    def maybe_escalate(self, disagreement: float, deliberation: HeterogeneousAgentDeliberation,
                        state: FinancialState, ctx: List[MemoryRecord]) -> Optional[AgentProposal]:
        if disagreement >= self.threshold:
            # Challenge-Agent gets a second, higher-weight pass
            return deliberation.propose("Challenge-Agent", state, ctx)
        return None


class DADelibMix4:
    def __init__(self):
        self.deliberation = HeterogeneousAgentDeliberation()
        self.analysis = AgreementDisagreementAnalysis()
        self.mixer = ConfidenceMixer()
        self.router = DynamicChallengeRouting()

    def decide(self, step_idx: int, subject_agent: str, state: FinancialState,
               ctx: List[MemoryRecord]) -> CollectiveDecisionState:
        proposals = [self.deliberation.propose(role, state, ctx) for role in AGENTS]
        agreement, disagreement, sim_matrix = self.analysis.analyze(proposals)

        challenge_triggered = False
        escalated = self.router.maybe_escalate(disagreement, self.deliberation, state, ctx)
        if escalated is not None:
            challenge_triggered = True
            # replace the base Challenge-Agent proposal with the escalated (higher scrutiny) one
            proposals = [escalated if p.agent_role == "Challenge-Agent" else p for p in proposals]
            escalated.confidence = min(0.99, escalated.confidence * 1.2)
            agreement, disagreement, sim_matrix = self.analysis.analyze(proposals)

        final_action, coll_conf, influence = self.mixer.mix(proposals, sim_matrix)

        return CollectiveDecisionState(
            step_idx=step_idx, subject_agent=subject_agent, final_action=final_action,
            collective_confidence=coll_conf, agreement=agreement, disagreement=disagreement,
            influence=influence, challenge_triggered=challenge_triggered, proposals=proposals,
        )


delib_mix4 = DADelibMix4()
collective_states: Dict[Tuple[str, int], CollectiveDecisionState] = {}

for s in states:
    ctx = experience_contexts[(s.agent_id, s.step_idx)]
    cds = delib_mix4.decide(s.step_idx, s.agent_id, s, ctx)
    collective_states[(s.agent_id, s.step_idx)] = cds

pd.DataFrame([
    dict(agent=k[0], step=k[1], final_action=v.final_action,
         collective_conf=round(v.collective_confidence, 3),
         agreement=round(v.agreement, 2), disagreement=round(v.disagreement, 2),
         challenge=v.challenge_triggered)
    for k, v in list(collective_states.items())[:6]
])


---
## Phase 4 — 3.6 TempRel-Lite: Temporal Behaviour Interaction Modelling

The 4 agent roles become **nodes**; typed edges (agreement / disagreement / recommendation /
influence / challenge / information-exchange) connect them at every step. A **Relation Gate**
weights edge types by relevance, **Temporal Graph Attention** propagates a representation
across consecutive decision periods (simple exponential memory / attention over the last
`W` graphs), and a **Low-Rank Projection** compresses the result to a compact behaviour
vector per agent per step.


In [ ]:

EDGE_TYPES = ["agreement", "disagreement", "recommendation", "influence", "challenge", "info_exchange"]

class TemporalBehaviourGraphBuilder:
    def build_step_graph(self, cds: CollectiveDecisionState) -> nx.MultiDiGraph:
        G = nx.MultiDiGraph()
        for p in cds.proposals:
            G.add_node(p.agent_role, confidence=p.confidence, action=p.action)

        roles = [p.agent_role for p in cds.proposals]
        for i, a in enumerate(roles):
            for j, b in enumerate(roles):
                if a == b:
                    continue
                same_action = cds.proposals[i].action == cds.proposals[j].action
                etype = "agreement" if same_action else "disagreement"
                G.add_edge(a, b, type=etype, weight=1.0)
                G.add_edge(a, b, type="info_exchange", weight=0.3)
                G.add_edge(a, b, type="recommendation", weight=cds.proposals[i].confidence)

        # influence edges: everyone -> final decision "sink", weighted by mixer influence
        for role, w in cds.influence.items():
            G.add_edge(role, "DECISION", type="influence", weight=w)
        if cds.challenge_triggered:
            G.add_edge("Challenge-Agent", "DECISION", type="challenge", weight=1.0)
        return G


class RelationGate:
    """Learns (heuristically here) how much each edge type should matter."""
    DEFAULT_WEIGHTS = {"agreement": 0.9, "disagreement": 1.3, "recommendation": 0.7,
                        "influence": 1.4, "challenge": 1.6, "info_exchange": 0.4}

    def gate(self, G: nx.MultiDiGraph) -> Dict[Tuple[str, str, str], float]:
        gated = {}
        for u, v, data in G.edges(data=True):
            base_w = data["weight"]
            gated[(u, v, data["type"])] = base_w * self.DEFAULT_WEIGHTS[data["type"]]
        return gated


class TemporalGraphAttention:
    """Attention over the last `window` step-graphs: more recent graphs get exponentially
    higher weight (recency-decayed attention), producing a smoothed per-node embedding."""
    def __init__(self, window: int = 6, decay: float = 0.7):
        self.window = window
        self.decay = decay
        self.history: Dict[str, deque] = defaultdict(lambda: deque(maxlen=window))

    def node_embedding(self, role: str, G: nx.MultiDiGraph, gated_edges: Dict) -> np.ndarray:
        in_w = sum(w for (u, v, t), w in gated_edges.items() if v == role)
        out_w = sum(w for (u, v, t), w in gated_edges.items() if u == role)
        deg = G.degree(role) if role in G else 0
        conf = G.nodes[role].get("confidence", 0.5) if role in G else 0.5
        vec = np.array([in_w, out_w, float(deg), conf])
        self.history[role].append(vec)

        hist = list(self.history[role])
        weights = np.array([self.decay ** (len(hist) - 1 - i) for i in range(len(hist))])
        weights /= weights.sum()
        smoothed = np.average(np.stack(hist), axis=0, weights=weights)
        return smoothed


class LowRankProjection:
    def __init__(self, in_dim: int = 4, out_dim: int = 3, seed: int = 7):
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0, 1 / math.sqrt(in_dim), size=(in_dim, out_dim))

    def project(self, vec: np.ndarray) -> np.ndarray:
        return vec @ self.W


class TempRelLite:
    def __init__(self):
        self.builder = TemporalBehaviourGraphBuilder()
        self.gate = RelationGate()
        self.attention = TemporalGraphAttention()
        self.projection = LowRankProjection()

    def process_step(self, cds: CollectiveDecisionState) -> Dict[str, np.ndarray]:
        G = self.builder.build_step_graph(cds)
        gated_edges = self.gate.gate(G)
        reps = {}
        for role in AGENTS:
            emb = self.attention.node_embedding(role, G, gated_edges)
            reps[role] = self.projection.project(emb)
        return reps


temprel = TempRelLite()
behaviour_reps: Dict[Tuple[str, int], np.ndarray] = {}   # (role, step) -> compressed rep

steps_sorted = sorted(set(s.step_idx for s in states))
for step in steps_sorted:
    # one collective decision per subject agent per step; TempRel-Lite processes the graph
    # for each subject-agent's deliberation round at this step
    for subj in AGENTS:
        key = (subj, step)
        if key not in collective_states:
            continue
        reps = temprel.process_step(collective_states[key])
        for role, vec in reps.items():
            behaviour_reps[(subj, role, step)] = vec

print(f"Computed {len(behaviour_reps)} temporal behaviour representations "
      f"(subject x role x step), dim={next(iter(behaviour_reps.values())).shape[0]}")


---
## Phase 5 — 3.7 DriftCont-X: Behavioural Drift & Decision Contagion

**Behavioural drift**: track `Δ` between consecutive behavioural representations for the
*subject agent's own role slice*; run **online concept-drift detection** (ADWIN-style
windowed mean-shift test) plus **temporal change-point detection** (CUSUM) on the drift
signal (strategy, risk preference, confidence, MRD, decision consistency).

**Decision contagion**: when agent A drifts, scan the temporal interaction graph for edges
into agent B; test whether B's behavioural change follows A's in time, in the same
direction, with material influence weight — i.e. causal propagation, not coincidence.


In [ ]:

@dataclass
class DriftSignal:
    agent: str
    step: int
    drift_magnitude: float
    is_change_point: bool
    is_online_drift: bool


class OnlineConceptDrift:
    """Simple ADWIN-style detector: compares mean of a recent window vs an older window;
    flags drift when the shift exceeds `threshold` standard errors."""
    def __init__(self, window: int = 10, threshold: float = 2.0):
        self.window = window
        self.threshold = threshold
        self.buffers: Dict[str, deque] = defaultdict(lambda: deque(maxlen=window * 2))

    def update(self, agent: str, value: float) -> bool:
        buf = self.buffers[agent]
        buf.append(value)
        if len(buf) < self.window * 2:
            return False
        old, new = list(buf)[: self.window], list(buf)[self.window:]
        old_mean, new_mean = np.mean(old), np.mean(new)
        pooled_std = np.std(old + new) + 1e-9
        z = abs(new_mean - old_mean) / (pooled_std / math.sqrt(self.window))
        return z > self.threshold


class TemporalChangePointDetection:
    """CUSUM change-point detector on a scalar drift-magnitude series per agent."""
    def __init__(self, k: float = 0.3, h: float = 2.0):
        self.k, self.h = k, h
        self.pos: Dict[str, float] = defaultdict(float)
        self.neg: Dict[str, float] = defaultdict(float)
        self.mean: Dict[str, float] = defaultdict(float)
        self.n: Dict[str, int] = defaultdict(int)

    def update(self, agent: str, value: float) -> bool:
        self.n[agent] += 1
        n = self.n[agent]
        self.mean[agent] += (value - self.mean[agent]) / n
        dev = value - self.mean[agent]
        self.pos[agent] = max(0, self.pos[agent] + dev - self.k)
        self.neg[agent] = max(0, self.neg[agent] - dev - self.k)
        triggered = self.pos[agent] > self.h or self.neg[agent] > self.h
        if triggered:
            self.pos[agent] = 0.0
            self.neg[agent] = 0.0
        return triggered


class InfluencePropagationContagion:
    """Given a set of drift events and the per-step influence graph, tests whether one
    agent's drift propagates to another: temporal order + interaction direction +
    behavioural similarity + influence strength + downstream change."""
    def __init__(self, lag_window: int = 12, similarity_threshold: float = 0.6,
                 influence_threshold: float = 0.25):
        self.lag_window = lag_window
        self.sim_threshold = similarity_threshold
        self.influence_threshold = influence_threshold

    def detect(self, drift_events: List[DriftSignal],
               influence_by_step: Dict[int, Dict[str, float]],
               behaviour_reps: Dict[Tuple[str, str, int], np.ndarray]) -> List[Dict]:
        contagions = []
        events_by_agent = defaultdict(list)
        for e in drift_events:
            if e.is_change_point or e.is_online_drift:
                events_by_agent[e.agent].append(e)

        for source_agent, source_events in events_by_agent.items():
            for se in source_events:
                for target_agent, target_events in events_by_agent.items():
                    if target_agent == source_agent:
                        continue
                    for te in target_events:
                        lag = te.step - se.step
                        if not (0 < lag <= self.lag_window):
                            continue
                        infl = influence_by_step.get(te.step, {}).get(source_agent, 0.0)
                        if infl < self.influence_threshold:
                            continue
                        v_src = behaviour_reps.get((source_agent, source_agent, se.step))
                        v_tgt = behaviour_reps.get((target_agent, target_agent, te.step))
                        if v_src is None or v_tgt is None:
                            continue
                        sim = cosine_similarity(v_src.reshape(1, -1), v_tgt.reshape(1, -1))[0, 0]
                        if sim < self.sim_threshold:
                            continue
                        amp = te.drift_magnitude / (se.drift_magnitude + 1e-3)
                        contagions.append(dict(
                            source=source_agent, target=target_agent,
                            source_step=se.step, target_step=te.step, lag=lag,
                            influence=infl, similarity=float(sim),
                            amplification=float(np.clip(amp, 0, 5)),
                        ))
        return contagions


class DriftContX:
    def __init__(self):
        self.online = OnlineConceptDrift()
        self.changepoint = TemporalChangePointDetection()
        self.contagion = InfluencePropagationContagion()

    def compute_drift_series(self, behaviour_reps, mrd_df) -> List[DriftSignal]:
        signals = []
        prev = {}
        for step in steps_sorted:
            for agent in AGENTS:
                vec = behaviour_reps.get((agent, agent, step))
                if vec is None:
                    continue
                magnitude = float(np.linalg.norm(vec - prev.get(agent, vec)))
                prev[agent] = vec
                is_online = self.online.update(agent, magnitude)
                is_cp = self.changepoint.update(agent, magnitude)
                signals.append(DriftSignal(agent=agent, step=step, drift_magnitude=magnitude,
                                            is_change_point=is_cp, is_online_drift=is_online))
        return signals

    def run(self, behaviour_reps, collective_states, mrd_df):
        drift_signals = self.compute_drift_series(behaviour_reps, mrd_df)

        influence_by_step = defaultdict(dict)
        for (subj, step), cds in collective_states.items():
            if subj != cds.subject_agent:
                continue
            for role, w in cds.influence.items():
                influence_by_step[step][role] = max(influence_by_step[step].get(role, 0), w)

        contagions = self.contagion.detect(drift_signals, influence_by_step, behaviour_reps)

        # Decision Contagion Amplification Factor (DCAF): mean amplification across detected events
        dcaf = float(np.mean([c["amplification"] for c in contagions])) if contagions else 1.0
        return drift_signals, contagions, dcaf


driftcont = DriftContX()
drift_signals, contagion_events, DCAF = driftcont.run(behaviour_reps, collective_states, mrd_df)

drift_df = pd.DataFrame([asdict(d) for d in drift_signals])
print(f"Drift signals computed: {len(drift_df)}")
print(f"Change points detected: {drift_df['is_change_point'].sum()} | "
      f"Online-drift flags: {drift_df['is_online_drift'].sum()}")
print(f"Decision Contagion events detected: {len(contagion_events)}")
print(f"Decision Contagion Amplification Factor (DCAF) = {DCAF:.3f}")
pd.DataFrame(contagion_events).head(10) if contagion_events else print("(none in this synthetic run yet)")


### 5.1 Visualising Behavioural Drift & Contagion

In [ ]:

fig, axes = plt.subplots(len(AGENTS), 1, figsize=(12, 9), sharex=True)
for ax, agent in zip(axes, AGENTS):
    sub = drift_df[drift_df.agent == agent]
    ax.plot(sub["step"], sub["drift_magnitude"], color="steelblue", lw=1.2, label="drift magnitude")
    cps = sub[sub.is_change_point]
    ax.scatter(cps["step"], cps["drift_magnitude"], color="crimson", zorder=5, s=25, label="change point")
    ax.axvline(raw_df.attrs["drift_step"], color="gray", ls="--", lw=0.8, alpha=0.6)
    ax.set_ylabel(agent, fontsize=9)
    ax.legend(loc="upper left", fontsize=7)
axes[-1].set_xlabel("decision step")
fig.suptitle("Behavioural Drift Magnitude per Agent (dashed line = injected drift onset)")
plt.tight_layout()
plt.show()

if contagion_events:
    cdf = pd.DataFrame(contagion_events)
    agent_y = {a: i for i, a in enumerate(AGENTS)}
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.set_yticks(list(agent_y.values()))
    ax.set_yticklabels(list(agent_y.keys()))
    ax.set_ylim(-0.5, len(AGENTS) - 0.5)
    for _, row in cdf.iterrows():
        ax.annotate("", xy=(row["target_step"], agent_y[row["target"]]),
                    xytext=(row["source_step"], agent_y[row["source"]]),
                    arrowprops=dict(arrowstyle="->", color="darkorange", alpha=0.5, lw=1.3))
    ax.set_title("Detected Decision-Contagion Propagation (source \u2192 target)")
    ax.set_xlabel("decision step")
    plt.tight_layout()
    plt.show()


---
## Phase 6 — 3.8 ReflectAdapt-X: Self-Reflection, Memory Evolution & Policy Adaptation

After each episode's outcome is known, the **Self-Reflection Module** asks: was the
retrieved experience relevant? Was the decision appropriate? Did another agent's influence
sway it inappropriately? Was confidence justified? This becomes a structured reflection that
(a) is written back into the eRAG memory (raising/lowering `reflection_quality` on the
records that were actually retrieved) and (b) drives a lightweight **Parameter-Efficient
Policy Adapter** that nudges `ConfidenceMixer.reliability` per role based on observed
reward/error — closing the adaptive loop back into Phase 2/3.

**RMFS** (Reflection-to-Memory Feedback Score) measures how strongly past reflections are
now shaping retrieval (tracked as the mean reflection_quality of retrieved memories over time).


In [ ]:

@dataclass
class Reflection:
    agent: str
    step: int
    experience_relevant: bool
    decision_appropriate: bool
    influenced_by_other: bool
    confidence_justified: bool
    text: str


class SelfReflectionModule:
    def reflect(self, state: FinancialState, cds: CollectiveDecisionState,
                ctx: List[MemoryRecord], realized_outcome: float) -> Reflection:
        decision_appropriate = (realized_outcome > 0) == (cds.final_action in ("BUY", "HOLD"))
        experience_relevant = bool(ctx) and np.mean([c.outcome for c in ctx]) * realized_outcome > 0
        top_influencer = max(cds.influence, key=cds.influence.get) if cds.influence else None
        influenced_by_other = bool(top_influencer and top_influencer != state.agent_id and
                                    cds.influence[top_influencer] > 0.4)
        confidence_justified = (cds.collective_confidence > 0.6) == (abs(realized_outcome) > 20)

        text = (f"[{state.agent_id}@{state.step_idx}] action={cds.final_action} "
                f"outcome={realized_outcome:.1f} appropriate={decision_appropriate} "
                f"exp_relevant={experience_relevant} infl_by={top_influencer}")
        return Reflection(agent=state.agent_id, step=state.step_idx,
                           experience_relevant=experience_relevant,
                           decision_appropriate=decision_appropriate,
                           influenced_by_other=influenced_by_other,
                           confidence_justified=confidence_justified, text=text)


class ParameterEfficientPolicyAdapter:
    """Nudges per-role reliability weights in the Confidence Mixer using a tiny
    reward-weighted update rule (analogous to a LoRA-style low-parameter-count update,
    but expressed directly on the small reliability vector for transparency/runnability)."""
    def __init__(self, mixer: ConfidenceMixer, lr: float = 0.03):
        self.mixer = mixer
        self.lr = lr

    def update(self, cds: CollectiveDecisionState, reward: float):
        for role, infl in cds.influence.items():
            current = self.mixer.reliability[role]
            grad = infl * reward
            self.mixer.reliability[role] = float(np.clip(current + self.lr * grad, 0.05, 2.0))


class ReflectAdaptX:
    def __init__(self, memory: EpisodicMemoryBuilder, mixer: ConfidenceMixer):
        self.reflector = SelfReflectionModule()
        self.adapter = ParameterEfficientPolicyAdapter(mixer)
        self.rmfs_history: List[float] = []

    def step(self, state: FinancialState, cds: CollectiveDecisionState,
             ctx: List[MemoryRecord], realized_outcome: float):
        refl = self.reflector.reflect(state, cds, ctx, realized_outcome)

        quality_delta = (0.1 if refl.decision_appropriate else -0.1) + \
                         (0.05 if refl.experience_relevant else -0.05)
        for rec in ctx:
            rec.reflection_quality = float(np.clip(rec.reflection_quality + quality_delta, 0.0, 1.0))
            rec.reflection = refl.text

        reward = np.tanh(realized_outcome / 100)
        self.adapter.update(cds, reward)

        rmfs = float(np.mean([c.reflection_quality for c in ctx])) if ctx else 0.5
        self.rmfs_history.append(rmfs)
        return refl


reflect_adapt = ReflectAdaptX(erag.memory, delib_mix4.mixer)
reflections = []

for s in states:
    ctx = experience_contexts[(s.agent_id, s.step_idx)]
    cds = collective_states.get((s.agent_id, s.step_idx))
    if cds is None:
        continue
    refl = reflect_adapt.step(s, cds, ctx, s.outcome)
    reflections.append(refl)

print(f"Generated {len(reflections)} reflections.")
print("Final adapted role reliabilities (ConfidenceMixer):")
for role, rel in delib_mix4.mixer.reliability.items():
    print(f"  {role:18s} -> {rel:.3f}")

rmfs_series = pd.Series(reflect_adapt.rmfs_history)
print(f"\nFinal Reflection-to-Memory Feedback Score (RMFS, rolling mean last 20): "
      f"{rmfs_series.tail(20).mean():.3f}")


---
## Phase 7 — 3.9 ConforRisk-X: Uncertainty-Aware Risk Orchestration

Combines everything into the **Autonomous Financial Behaviour Risk Index (AFBRI)**:

`AFBRI_t = w_D·D_t + w_C·C_t + w_I·I_t + w_A·A_t + w_U·U_t`,  `Σw = 1`

- `D_t` — behavioural drift (normalized drift magnitude)
- `C_t` — decision contagion (whether/how strongly this agent is a contagion target this step)
- `I_t` — influence received from other agents
- `A_t` — Decision Contagion Amplification Factor (DCAF), agent-local
- `U_t` — conformal uncertainty estimate

**Conformal Prediction** calibrates "is the current behavioural state consistent with the
agent's own historical distribution?" using a distribution-free nonconformity score against
a calibration window, giving a proper uncertainty estimate (not just a heuristic).

Interventions: **Low → normal**, **Moderate → additional verification**,
**High → challenge review / isolation**, **High + High uncertainty → human approval**.


In [ ]:

class ConformalUncertaintyEstimator:
    """Split-conformal-style: nonconformity score = |drift_magnitude - rolling median| /
    rolling MAD, compared against a calibration-window quantile. Returns a value in [0,1]
    where higher = more anomalous / less conformal to the agent's own recent history."""
    def __init__(self, calib_window: int = 20, alpha: float = 0.1):
        self.calib_window = calib_window
        self.alpha = alpha
        self.history: Dict[str, deque] = defaultdict(lambda: deque(maxlen=calib_window))

    def estimate(self, agent: str, value: float) -> float:
        hist = self.history[agent]
        if len(hist) < 5:
            u = 0.5   # not enough calibration data yet -> moderate default uncertainty
        else:
            median = np.median(hist)
            mad = np.median(np.abs(np.array(hist) - median)) + 1e-6
            nonconformity = abs(value - median) / mad
            calib_scores = np.abs(np.array(hist) - median) / mad
            q = np.quantile(calib_scores, 1 - self.alpha)
            u = float(np.clip(nonconformity / (q + 1e-6), 0, 1))
        hist.append(value)
        return u


class BehaviouralRiskFusion:
    def __init__(self, w_drift=0.30, w_contagion=0.20, w_influence=0.15,
                 w_amplification=0.20, w_uncertainty=0.15):
        total = w_drift + w_contagion + w_influence + w_amplification + w_uncertainty
        self.w = dict(D=w_drift / total, C=w_contagion / total, I=w_influence / total,
                       A=w_amplification / total, U=w_uncertainty / total)

    def fuse(self, D, C, I, A, U) -> float:
        w = self.w
        return float(w["D"] * D + w["C"] * C + w["I"] * I + w["A"] * A + w["U"] * U)


class AdaptiveIntervention:
    def decide(self, afbri: float, uncertainty: float) -> str:
        if afbri < 0.35:
            return "Low -> Normal autonomous operation"
        if afbri < 0.6:
            return "Moderate -> Additional agent verification"
        if afbri >= 0.6 and uncertainty >= 0.6:
            return "High + High uncertainty -> HUMAN APPROVAL REQUIRED"
        return "High -> Challenge review / agent isolation"


class ConforRiskX:
    def __init__(self):
        self.conformal = ConformalUncertaintyEstimator()
        self.fusion = BehaviouralRiskFusion()
        self.intervention = AdaptiveIntervention()

    def run(self, drift_df: pd.DataFrame, mrd_df: pd.DataFrame,
            contagion_events: List[Dict], dcaf_global: float,
            collective_states: Dict) -> pd.DataFrame:
        contagion_targets = defaultdict(list)   # (agent, step) -> [amplification,...]
        for c in contagion_events:
            contagion_targets[(c["target"], c["target_step"])].append(c["amplification"])

        drift_norm = drift_df.copy()
        for agent, g in drift_norm.groupby("agent"):
            mx = g["drift_magnitude"].max() or 1.0
            drift_norm.loc[g.index, "drift_norm"] = g["drift_magnitude"] / mx

        mrd_lookup = {(r.agent, r.step): r.mrd for r in mrd_df.itertuples()}
        infl_lookup = defaultdict(dict)
        for (subj, step), cds in collective_states.items():
            if subj != cds.subject_agent:
                continue
            infl_lookup[(subj, step)] = cds.influence

        rows = []
        for r in drift_norm.itertuples():
            agent, step = r.agent, r.step
            D = float(r.drift_norm)
            C = min(1.0, len(contagion_targets.get((agent, step), [])) / 2)
            I = float(sum(v for role, v in infl_lookup.get((agent, step), {}).items()
                          if role != agent))
            I = min(1.0, I)
            A = min(1.0, np.mean(contagion_targets[(agent, step)]) / 3) if contagion_targets.get((agent, step)) else min(1.0, dcaf_global / 3)
            U = self.conformal.estimate(agent, r.drift_magnitude)

            afbri = self.fusion.fuse(D, C, I, A, U)
            action = self.intervention.decide(afbri, U)

            rows.append(dict(agent=agent, step=step, D=D, C=C, I=I, A=A, U=U,
                              AFBRI=afbri, intervention=action,
                              MRD=mrd_lookup.get((agent, step), np.nan)))
        return pd.DataFrame(rows)


conforrisk = ConforRiskX()
risk_df = conforrisk.run(drift_df, mrd_df, contagion_events, DCAF, collective_states)
risk_df.groupby("agent")["AFBRI"].agg(["mean", "max"]).round(3)


### 7.1 Visualising AFBRI & Interventions

In [ ]:

fig, ax = plt.subplots(figsize=(12, 4.5))
for agent in AGENTS:
    sub = risk_df[risk_df.agent == agent]
    ax.plot(sub["step"], sub["AFBRI"], label=agent, lw=1.3)
ax.axhline(0.35, color="gold", ls="--", lw=1, label="Low/Moderate threshold")
ax.axhline(0.6, color="crimson", ls="--", lw=1, label="Moderate/High threshold")
ax.axvline(raw_df.attrs["drift_step"], color="gray", ls=":", lw=0.8)
ax.set_title("Autonomous Financial Behaviour Risk Index (AFBRI) over time")
ax.set_xlabel("decision step"); ax.set_ylabel("AFBRI")
ax.legend(loc="upper left", fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

print("Intervention level counts (all agents, all steps):")
print(risk_df["intervention"].value_counts())

print("\nSteps requiring HUMAN APPROVAL:")
risk_df[risk_df["intervention"].str.contains("HUMAN")][["agent", "step", "AFBRI", "U", "D", "C"]].head(15)


---
## End-to-End Orchestrator

Wraps all seven phases into a single `BehaviouralDriftContagionPipeline` object so the
whole framework can be re-run (e.g. on real `AI-Trader` exports) with one call:
`pipeline.run(raw_df)` → returns the final `risk_df` plus every intermediate artifact for
inspection/auditing.


In [ ]:

class BehaviouralDriftContagionPipeline:
    """End-to-end: raw AI-Trader-style logs -> AFBRI risk table + full audit trail."""

    def __init__(self):
        self.preprocessor = FinStateLitePreprocessor()
        self.erag = eRAGReflectX(top_k=8)
        self.delib = DADelibMix4()
        self.temprel = TempRelLite()
        self.driftcont = DriftContX()
        self.reflect_adapt = ReflectAdaptX(self.erag.memory, self.delib.mixer)
        self.conforrisk = ConforRiskX()

    def run(self, raw_df: pd.DataFrame) -> Dict[str, Any]:
        # Phase 1
        states = self.preprocessor.run(raw_df)

        # Phases 2-3 (interleaved: retrieve -> decide -> grow memory, causally per step)
        experience_contexts, mrd_rows, collective_states = {}, [], {}
        for s in states:
            ctx, mrd = self.erag.retrieve(s)
            experience_contexts[(s.agent_id, s.step_idx)] = ctx
            mrd_rows.append(dict(agent=s.agent_id, step=s.step_idx, mrd=mrd, regime=s.market_regime))
            cds = self.delib.decide(s.step_idx, s.agent_id, s, ctx)
            collective_states[(s.agent_id, s.step_idx)] = cds
            self.erag.observe(s)
        mrd_df = pd.DataFrame(mrd_rows)

        # Phase 4
        behaviour_reps = {}
        steps_sorted = sorted(set(s.step_idx for s in states))
        for step in steps_sorted:
            for subj in AGENTS:
                key = (subj, step)
                if key not in collective_states:
                    continue
                for role, vec in self.temprel.process_step(collective_states[key]).items():
                    behaviour_reps[(subj, role, step)] = vec

        # Phase 5
        drift_signals, contagion_events, dcaf = self.driftcont.run(
            behaviour_reps, collective_states, mrd_df)
        drift_df = pd.DataFrame([asdict(d) for d in drift_signals])

        # Phase 6 (reflection + adaptation)
        for s in states:
            ctx = experience_contexts[(s.agent_id, s.step_idx)]
            cds = collective_states.get((s.agent_id, s.step_idx))
            if cds is not None:
                self.reflect_adapt.step(s, cds, ctx, s.outcome)

        # Phase 7
        risk_df = self.conforrisk.run(drift_df, mrd_df, contagion_events, dcaf, collective_states)

        return dict(states=states, mrd_df=mrd_df, collective_states=collective_states,
                    behaviour_reps=behaviour_reps, drift_df=drift_df,
                    contagion_events=contagion_events, dcaf=dcaf, risk_df=risk_df)


# Fresh end-to-end run from raw data, using only the orchestrator:
pipeline = BehaviouralDriftContagionPipeline()
results = pipeline.run(load_ai_trader_data(n_steps=180))
print("Pipeline complete.")
print(results["risk_df"].groupby("agent")["AFBRI"].mean().round(3))


---
## Notes on Swapping in Real `AI-Trader` Data

1. Clone `github.com/HKUDS/AI-Trader` and point a loader at its exported observation /
   action / portfolio logs; map its columns onto `RAW_COLUMNS` (rename/derive as needed).
2. Replace `load_ai_trader_data()` with that loader — nothing else changes, since every
   downstream Phase only depends on the `RAW_COLUMNS` schema.
3. For production use, replace the rule-based `HeterogeneousAgentDeliberation.propose()`
   heuristics with actual learned/LLM-backed agent policies (Market/Risk/Portfolio/Challenge),
   keeping the same `AgentProposal` interface.
4. `FinStateLitePreprocessor.fit_normalizers()` must be fit on a held-out **train** slice only
   (already enforced) to avoid look-ahead leakage when you swap in real, longer histories.
5. Tune `ConforRisk-X` weights (`w_D, w_C, w_I, w_A, w_U`) and thresholds against a labeled
   validation set of known-anomalous trading episodes once available.
